# CP-SAT Validation & Benchmark (Kaggle / Colab)

OR-Tools' CP-SAT solver hangs on macOS 15.x with ortools 9.15.x, so this notebook runs the validation on Linux (Kaggle / Colab) instead.

**What this notebook does:**
1. Install OR-Tools + repo dependencies
2. Clone the repo (or use a mounted path)
3. Sanity-check the CP-SAT solver on a trivial 1-var problem
4. Direct CP-SAT smoke test on a 5-pallet 40HC voyage
5. Run the full pytest suite for `test_cpsat.py`
6. Run `scripts/benchmark_cpsat.py` — CP-SAT vs 5 heuristics vs GA
7. Print the CSV summary

## 1. Install dependencies

In [1]:
!pip install -q 'ortools>=9.10,<10' \
  'pydantic==2.9.2' 'pydantic-settings==2.6.1' \
  fastapi 'uvicorn[standard]' python-multipart websockets orjson loguru \
  numpy gymnasium deap shapely pandas \
  'pytest>=8' pytest-asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.9/434.9 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.9/866.9 kB 46.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.9.2 which is incompatible.
googl

## 2. Locate / clone the repo

Pinned to branch `cpsat-baseline` (change `BRANCH` to `'main'` once merged).

The cell below tries common locations (Kaggle dataset, `/kaggle/working`, `/content`, `./`) and looks for `app/algorithms/cpsat.py` as a sentinel. If no clone has it, a fresh shallow clone of the branch happens. Safe to re-run after a Colab reconnect — stale clones missing the sentinel are wiped first.

For a **private** repo, set `GITHUB_TOKEN` first (Kaggle: *Add-ons → Secrets*; Colab: `from google.colab import userdata; os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')`).

In [2]:
import os, subprocess, sys, pathlib, shutil

REPO_URL = 'https://github.com/Seif-Sameh/loading-service-2.git'
BRANCH = 'cpsat-baseline'   # change to 'main' once this branch is merged
SENTINEL = 'app/algorithms/cpsat.py'  # file proving the clone has the CP-SAT code

candidates = [
    '/kaggle/working/loading-service-2',
    '/content/loading-service-2',
    './loading-service-2',
]
REPO_DIR = next((p for p in candidates if pathlib.Path(p, SENTINEL).exists()), None)

if REPO_DIR is None:
    # Either nothing cloned, or a stale clone on a branch missing cpsat.py.
    # Pick a writable target and (re)clone.
    target = '/kaggle/working/loading-service-2' if pathlib.Path('/kaggle/working').exists() else '/content/loading-service-2'
    if pathlib.Path(target).exists():
        shutil.rmtree(target)
    # Also check a read-only Kaggle dataset mount
    ro = pathlib.Path('/kaggle/input/loading-service-2')
    if ro.exists() and (ro / SENTINEL).exists():
        shutil.copytree(ro, target)
    else:
        tok = os.environ.get('GITHUB_TOKEN', '').strip()
        url = REPO_URL.replace('https://', f'https://{tok}@') if tok else REPO_URL
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, target])
    REPO_DIR = target

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('REPO_DIR =', REPO_DIR)
print('has', SENTINEL, ':', pathlib.Path(REPO_DIR, SENTINEL).exists())
print('contents:', sorted(os.listdir(REPO_DIR))[:20])

REPO_DIR = /content/loading-service-2
has app/algorithms/cpsat.py : True
contents: ['.env.example', '.git', '.gitignore', 'Dockerfile', 'Evals', 'Makefile', 'README.md', 'app', 'data', 'docker-compose.yml', 'models', 'notebooks', 'pyproject.toml', 'requirements-dev.txt', 'requirements-rl.txt', 'requirements.txt', 'scripts', 'tests']


## 3. OR-Tools sanity — does CP-SAT even solve `x >= 5`?

Locally this hangs forever on macOS. On Linux this should finish in <0.1s.

In [3]:
import time
from ortools.sat.python import cp_model
m = cp_model.CpModel()
x = m.NewIntVar(0, 10, 'x')
m.Add(x >= 5)
m.Maximize(x)
s = cp_model.CpSolver()
s.parameters.max_time_in_seconds = 5
t0 = time.perf_counter()
status = s.Solve(m)
print(f'status={s.StatusName(status)}  x={s.Value(x)}  in {time.perf_counter()-t0:.3f}s')
assert status == cp_model.OPTIMAL and s.Value(x) == 10, 'OR-Tools install broken'

status=OPTIMAL  x=10  in 0.007s


## 4. Direct CP-SAT smoke test — 5 pallets in a 40HC

Calls `_solve_cpsat` directly with synthetic items. Bypasses the env/select replay so we can be certain the solver itself works before running the full benchmark.

In [4]:
import os, sys, pathlib, time
# Self-contained: re-establish REPO_DIR after a possible kernel restart.
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand
        os.chdir(REPO_DIR)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        break

from app.algorithms.cpsat import CPSATConfig, _solve_cpsat
from app.catalog.loader import get_container, get_cargo_preset

container = get_container('40HC')
items = [get_cargo_preset('eur_pallet_light', item_id=f'p{i}') for i in range(5)]
cfg = CPSATConfig(time_limit_s=20.0, num_search_workers=2, grid_mm=100, log_search_progress=True)
t0 = time.perf_counter()
placements, status, obj = _solve_cpsat(container, items, cfg)
print(f'\nstatus={status}  planned={len(placements)}/{len(items)}  obj={obj}  time={time.perf_counter()-t0:.2f}s')
for p in placements:
    print(f'  {p.item_id}: pos=({p.position.x_mm},{p.position.y_mm},{p.position.z_mm})  rot={p.rotation}')


status=OPTIMAL  planned=5/5  obj=5760  time=0.03s
  p0: pos=(1200,100,0)  rot=0
  p1: pos=(0,1300,1200)  rot=0
  p2: pos=(2400,100,0)  rot=1
  p3: pos=(0,1300,0)  rot=1
  p4: pos=(0,100,0)  rot=0


## 5. Unit tests

In [5]:
import os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
!cd {REPO_DIR} && python -m pytest tests/test_cpsat.py -v --tb=short 2>&1 | tail -40

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/loading-service-2
configfile: pyproject.toml
plugins: asyncio-1.3.0, typeguard-4.5.1, anyio-4.13.0, langsmith-0.7.34
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... collected 4 items

tests/test_cpsat.py::test_cpsat_packs_small_voyage PASSED                [ 25%]
tests/test_cpsat.py::test_cpsat_beats_or_ties_bottom_left_on_mixed_bag PASSED [ 50%]
tests/test_cpsat.py::test_cpsat_registry_access PASSED                   [ 75%]
tests/test_cpsat.py::test_cpsat_respects_reefer_constraint PASSED        [100%]

============================== 4 passed in 1.02s ===============================


## 6. Full benchmark — CP-SAT vs heuristics vs GA

Tweak `--voyages`, `--items`, `--cpsat-time` for budget. Defaults: 5 voyages × 30 items, 30s CP-SAT budget per voyage (≈2.5 min total for CP-SAT alone).

In [6]:
import os, sys, pathlib, time, csv, statistics
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand
        os.chdir(REPO_DIR)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        break

from app.algorithms import get_algorithm
from app.algorithms.base import solve
from app.catalog.loader import get_container
from app.data.alexandria_sampler import AlexandriaSampler, SamplerConfig

# --- knobs ---
VOYAGES, ITEMS, CONTAINER, SEED = 5, 30, '40HC', 42
CPSAT_TIME, CPSAT_WORKERS = 30.0, 4
MORL_CKPT = ''   # set to a .pt path to include MORL-PCT
# -------------

sampler = AlexandriaSampler(SamplerConfig(n_items=ITEMS, strategy='mixed', seed=SEED))
cont = get_container(CONTAINER)
voyages = [(cont, sampler.sample()) for _ in range(VOYAGES)]

algo_codes = [
    ('bl', {}), ('extreme_points', {}), ('baf', {}), ('bssf', {}), ('blsf', {}),
    ('ga', {}),
    ('cpsat', {'time_limit_s': CPSAT_TIME, 'num_search_workers': CPSAT_WORKERS, 'enforce_imdg': True}),
]
if MORL_CKPT and pathlib.Path(MORL_CKPT).exists():
    algo_codes.append(('morl_pct', {'weights_path': MORL_CKPT, 'preference': [0.7, 0.1, 0.1, 0.05, 0.05]}))


def _ae(placements, container):
    if not placements:
        return 0.0
    door = 0.20 * container.internal.length_mm
    return sum(
        1 for p in placements
        if (p.position.x_mm + p.rotated_dimensions.length_mm / 2) <= door
    ) / len(placements)


rows = []
print(f'Building voyage suite: {VOYAGES} × {ITEMS} items, container={CONTAINER}')
for vi, (c, items) in enumerate(voyages):
    print(f'\n=== voyage {vi+1}/{VOYAGES}  ({len(items)} items) ===')
    for code, kwargs in algo_codes:
        algo = get_algorithm(code, **kwargs)
        t0 = time.perf_counter()
        if hasattr(algo, 'prepare'):
            algo.prepare(c, items)
        res, _ = solve(algorithm=algo, container=c, items=items)
        elapsed = time.perf_counter() - t0
        k = res.kpis
        ae = _ae(res.placements, c)
        ss = max(0.0, (len(res.placements) - k.unstable_count) / max(len(res.placements), 1))
        cps_status = str(algo.meta.get('cpsat_status', '')) if code == 'cpsat' else ''
        rows.append({
            'voyage': vi,
            'algorithm': code,
            'util_pct': 100 * k.utilization,
            'placed_pct': 100 * len(res.placements) / max(len(items), 1),
            'access_eff': ae,
            'stability_score': ss,
            'cog_long_abs': abs(k.cog_long_dev),
            'weight_pct': 100 * k.weight_used,
            'elapsed_s': elapsed,
            'cpsat_status': cps_status,
        })
        r = rows[-1]
        print(
            f"  {code:<16} util {r['util_pct']:>6.2f}%  placed {r['placed_pct']:>6.2f}%  "
            f"AE {ae:>5.2f}  SS {ss:>5.2f}  t {elapsed:>6.2f}s"
        )

# Aggregate
print('\n\n=== AGGREGATE (mean across voyages) ===')
print(
    f'{"algorithm":<16} {"util%":>7} {"std":>5} {"placed%":>8} {"AE":>5} {"SS":>5} '
    f'{"|CoG|":>6} {"wt%":>5} {"s":>6}'
)
print('-' * 75)
by = {}
for r in rows:
    by.setdefault(r['algorithm'], []).append(r)
for code, _ in algo_codes:
    rs = by.get(code, [])
    if not rs:
        continue
    utils = [r['util_pct'] for r in rs]
    u_std = statistics.stdev(utils) if len(utils) > 1 else 0.0
    print(
        f"{code:<16} {statistics.fmean(utils):>7.2f} {u_std:>5.2f} "
        f"{statistics.fmean(r['placed_pct'] for r in rs):>8.2f} "
        f"{statistics.fmean(r['access_eff'] for r in rs):>5.2f} "
        f"{statistics.fmean(r['stability_score'] for r in rs):>5.2f} "
        f"{statistics.fmean(r['cog_long_abs'] for r in rs):>6.3f} "
        f"{statistics.fmean(r['weight_pct'] for r in rs):>5.2f} "
        f"{statistics.fmean(r['elapsed_s'] for r in rs):>6.2f}"
    )

# CSV — each statement on its own line so paste-mangling can't break it.
out_dir = pathlib.Path(REPO_DIR, 'benchmarks/out')
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / f"cpsat_benchmark_{time.strftime('%Y%m%d_%H%M%S')}.csv"
with csv_path.open('w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
print(f'\n→ wrote {csv_path}')

Building voyage suite: 5 × 30 items, container=40HC

=== voyage 1/5  (30 items) ===
  bl               util  11.76%  placed  86.67%  AE  0.54  SS  1.00  t   0.20s
  extreme_points   util  11.76%  placed  86.67%  AE  0.50  SS  0.96  t   0.32s
  baf              util  16.01%  placed  90.00%  AE  0.19  SS  0.78  t   0.09s
  bssf             util  11.76%  placed  86.67%  AE  0.65  SS  0.81  t   0.08s
  blsf             util  16.01%  placed  90.00%  AE  0.07  SS  0.96  t   0.29s
  ga               util  11.76%  placed  86.67%  AE  0.54  SS  1.00  t 213.53s
  cpsat            util  16.01%  placed  90.00%  AE  0.30  SS  0.89  t   0.55s [OPTIMAL]

=== voyage 2/5  (30 items) ===
  bl               util   5.02%  placed  63.33%  AE  1.00  SS  0.95  t   0.09s
  extreme_points   util   4.84%  placed  56.67%  AE  0.94  SS  0.88  t   0.09s
  baf              util   4.87%  placed  60.00%  AE  0.94  SS  0.94  t   0.07s
  bssf             util   9.27%  placed  66.67%  AE  0.80  SS  0.95  t   0.09s
  bls

## 7. Show the CSV

In [7]:
import pandas as pd, glob, os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
csvs = sorted(glob.glob(os.path.join(REPO_DIR, 'benchmarks/out/cpsat_benchmark_*.csv')))
if not csvs:
    print('No CSV found — did the benchmark run?')
else:
    df = pd.read_csv(csvs[-1])
    print('latest:', csvs[-1])
    summary = df.groupby('algorithm').agg(
        util_mean=('util_pct', 'mean'),
        util_std=('util_pct', 'std'),
        placed_mean=('placed_pct', 'mean'),
        access_eff=('access_eff', 'mean'),
        stability=('stability_score', 'mean'),
        cog=('cog_long_abs', 'mean'),
        time_s=('elapsed_s', 'mean'),
    ).round(2).sort_values('util_mean', ascending=False)
    print(summary)

latest: /content/loading-service-2/benchmarks/out/cpsat_benchmark_20260518_202730.csv
                util_mean  util_std  placed_mean  access_eff  stability   cog  \
algorithm                                                                       
cpsat               11.47      4.81        87.33        0.40       0.95  0.20   
blsf                10.62      4.73        86.67        0.25       0.98  0.12   
bssf                 9.77      3.81        86.00        0.68       0.81  0.33   
extreme_points       8.88      4.43        84.00        0.68       0.90  0.33   
bl                   8.62      3.96        84.67        0.71       0.98  0.33   
ga                   8.62      3.96        84.67        0.71       0.98  0.33   
baf                  8.50      4.79        84.00        0.57       0.82  0.28   

                time_s  
algorithm               
cpsat             0.58  
blsf              0.29  
bssf              0.12  
extreme_points    0.28  
bl                0.23  
ga       

---

## 8. Full evaluation suite (edge cases + PCT + MORL-PCT)

Stress-tests every algorithm across **6 voyage profiles** × multiple seeds:

| suite          | items | container | content                                  | what it stresses                |
|----------------|------:|-----------|------------------------------------------|---------------------------------|
| `small_easy`   |    12 | 40HC      | EUR pallets (preset)                     | CP-SAT should hit OPTIMAL fast  |
| `medium_mixed` |    30 | 40HC      | Wadaboa+presets mixed                    | baseline (matches §6)           |
| `large_mixed`  |    60 | 40HC      | Wadaboa+presets mixed                    | heuristics start to lose more   |
| `20gp_tight`   |    20 | 20GP      | mixed                                    | smaller envelope → density matters |
| `hazmat_heavy` |    24 | 40HC      | mix of class-3 drums, class-8 corrosives, plus filler | IMDG segregation        |
| `reefer`       |    12 | 20RF      | reefer fruit pallets only                | reefer container path           |

Upload your checkpoints to Colab (`/content/...`) and set the paths in the **next cell**. Leave a path empty (`''`) to skip that algorithm.

In [1]:
# ========== CONFIG — edit these ==========
PCT_CKPT  = '/content/pct_latest_5.pt'              # '' to skip PCT
MORL_CKPT = '/content/morl_pct_latest.pt'         # '' to skip MORL-PCT
MORL_PREFERENCE = [0.7, 0.1, 0.1, 0.05, 0.05]     # util / access / stability / cog / weight

SEEDS = [42, 7, 123]                              # one full sweep per seed
CPSAT_TIME = 30.0                                 # seconds per voyage for CP-SAT
CPSAT_WORKERS = 4
RUN_GA = True                                     # GA is ~200s/voyage — set False to skip
SUITES_TO_RUN = [
    'small_easy', 'medium_mixed', 'large_mixed',
    '20gp_tight', 'hazmat_heavy', 'reefer',
]
# =========================================

import os, sys, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand
        os.chdir(REPO_DIR)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        break

# Install torch only if a neural checkpoint is requested
_need_torch = bool(PCT_CKPT) or bool(MORL_CKPT)
if _need_torch:
    try:
        import torch  # noqa: F401
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])

# Sanity report
print('REPO_DIR    =', REPO_DIR)
print('PCT_CKPT    =', PCT_CKPT, '(exists)' if PCT_CKPT and pathlib.Path(PCT_CKPT).exists() else '(SKIP)')
print('MORL_CKPT   =', MORL_CKPT, '(exists)' if MORL_CKPT and pathlib.Path(MORL_CKPT).exists() else '(SKIP)')
print('SEEDS       =', SEEDS)
print('SUITES      =', SUITES_TO_RUN)
print('CPSAT_TIME  =', CPSAT_TIME, 's   workers =', CPSAT_WORKERS)
print('RUN_GA      =', RUN_GA)

REPO_DIR    = /content/loading-service-2
PCT_CKPT    = /content/pct_latest_5.pt (exists)
MORL_CKPT   = /content/morl_pct_latest.pt (exists)
SEEDS       = [42, 7, 123]
SUITES      = ['small_easy', 'medium_mixed', 'large_mixed', '20gp_tight', 'hazmat_heavy', 'reefer']
CPSAT_TIME  = 30.0 s   workers = 4
RUN_GA      = True


In [2]:
import random
from app.catalog.loader import get_container, get_cargo_preset
from app.data.alexandria_sampler import AlexandriaSampler, SamplerConfig

# Suite specs — kept minimal; tweak counts here if a sweep is too slow.
SUITE_SPECS = {
    'small_easy':   dict(items=12, container='40HC', kind='presets',  preset='eur_pallet_light'),
    'medium_mixed': dict(items=30, container='40HC', kind='sampler',  strategy='mixed'),
    'large_mixed':  dict(items=60, container='40HC', kind='sampler',  strategy='mixed'),
    '20gp_tight':   dict(items=20, container='20GP', kind='sampler',  strategy='mixed'),
    'hazmat_heavy': dict(items=24, container='40HC', kind='hazmat'),
    'reefer':       dict(items=12, container='20RF', kind='presets',  preset='reefer_fruit_pallet'),
}

HAZMAT_MIX = ['steel_drum_200l', 'hazmat_corrosive_drum', 'carton_small', 'eur_pallet_light']


def build_voyage(suite_name, seed):
    cfg = SUITE_SPECS[suite_name]
    cont = get_container(cfg['container'])
    if cfg['kind'] == 'presets':
        items = [
            get_cargo_preset(cfg['preset'], item_id=f'{suite_name[:3]}-{seed}-{i:03d}')
            for i in range(cfg['items'])
        ]
    elif cfg['kind'] == 'hazmat':
        rng = random.Random(seed)
        items = [
            get_cargo_preset(rng.choice(HAZMAT_MIX), item_id=f'haz-{seed}-{i:03d}')
            for i in range(cfg['items'])
        ]
    else:  # sampler
        s = AlexandriaSampler(SamplerConfig(
            n_items=cfg['items'], strategy=cfg['strategy'], seed=seed,
        ))
        items = s.sample()
    return cont, items


# Quick sanity print
for name in SUITES_TO_RUN:
    cont, items = build_voyage(name, SEEDS[0])
    print(f'{name:<14} container={cont.code.value}  items={len(items)}  '
          f'first_id={items[0].id if items else "—"}')

small_easy     container=40HC  items=12  first_id=sma-42-000
medium_mixed   container=40HC  items=30  first_id=alex-0000
large_mixed    container=40HC  items=60  first_id=alex-0000
20gp_tight     container=20GP  items=20  first_id=alex-0000
hazmat_heavy   container=40HC  items=24  first_id=haz-42-000
reefer         container=20RF  items=12  first_id=ree-42-000


In [3]:
import time, csv, statistics
from app.algorithms import get_algorithm
from app.algorithms.base import solve


def _ae(placements, container):
    if not placements:
        return 0.0
    door = 0.20 * container.internal.length_mm
    return sum(
        1 for p in placements
        if (p.position.x_mm + p.rotated_dimensions.length_mm / 2) <= door
    ) / len(placements)


def _algo_specs():
    specs = [
        ('bl', {}),
        ('extreme_points', {}),
        ('baf', {}),
        ('bssf', {}),
        ('blsf', {}),
        ('cpsat', {'time_limit_s': CPSAT_TIME, 'num_search_workers': CPSAT_WORKERS, 'enforce_imdg': True}),
    ]
    if RUN_GA:
        specs.insert(5, ('ga', {}))
    if PCT_CKPT and pathlib.Path(PCT_CKPT).exists():
        specs.append(('pct', {'weights_path': PCT_CKPT}))
    if MORL_CKPT and pathlib.Path(MORL_CKPT).exists():
        specs.append(('morl_pct', {'weights_path': MORL_CKPT, 'preference': MORL_PREFERENCE}))
    return specs


def run_one(code, kwargs, cont, items):
    algo = get_algorithm(code, **kwargs)
    t0 = time.perf_counter()
    if hasattr(algo, 'prepare'):
        algo.prepare(cont, items)
    res, _ = solve(algorithm=algo, container=cont, items=items)
    elapsed = time.perf_counter() - t0
    k = res.kpis
    return {
        'util_pct': 100 * k.utilization,
        'placed_pct': 100 * len(res.placements) / max(len(items), 1),
        'access_eff': _ae(res.placements, cont),
        'stability_score': max(0.0, (len(res.placements) - k.unstable_count) / max(len(res.placements), 1)),
        'cog_long_abs': abs(k.cog_long_dev),
        'weight_pct': 100 * k.weight_used,
        'elapsed_s': elapsed,
        'cpsat_status': str(algo.meta.get('cpsat_status', '')) if code == 'cpsat' else '',
    }


algo_specs = _algo_specs()
print('Algorithms:', [c for c, _ in algo_specs])
print('Total runs:', len(SUITES_TO_RUN) * len(SEEDS) * len(algo_specs))

eval_rows = []
t_global = time.perf_counter()
for suite_name in SUITES_TO_RUN:
    spec = SUITE_SPECS[suite_name]
    print(f'\n========== {suite_name}  ({spec["container"]}, {spec["items"]} items) ==========')
    for seed in SEEDS:
        cont, items = build_voyage(suite_name, seed)
        print(f'-- seed {seed} --')
        for code, kwargs in algo_specs:
            try:
                m = run_one(code, kwargs, cont, items)
            except Exception as e:
                print(f'  {code:<16} FAILED: {type(e).__name__}: {e}')
                continue
            row = {'suite': suite_name, 'seed': seed, 'algorithm': code, **m}
            eval_rows.append(row)
            print(
                f'  {code:<16} util {m["util_pct"]:>6.2f}%  placed {m["placed_pct"]:>6.2f}%  '
                f'AE {m["access_eff"]:>4.2f}  SS {m["stability_score"]:>4.2f}  '
                f't {m["elapsed_s"]:>7.2f}s'
            )

print(f'\nTotal eval wall time: {(time.perf_counter() - t_global)/60:.1f} min')

# Save master CSV
out_dir = pathlib.Path(REPO_DIR, 'benchmarks/out')
out_dir.mkdir(parents=True, exist_ok=True)
master_csv = out_dir / f"full_eval_{time.strftime('%Y%m%d_%H%M%S')}.csv"
with master_csv.open('w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=list(eval_rows[0].keys()))
    w.writeheader()
    w.writerows(eval_rows)
print(f'→ wrote {master_csv}')

Algorithms: ['bl', 'extreme_points', 'baf', 'bssf', 'blsf', 'ga', 'cpsat', 'pct', 'morl_pct']
Total runs: 162

========== small_easy  (40HC, 12 items) ==========
-- seed 42 --
  bl               util  18.11%  placed 100.00%  AE 0.33  SS 1.00  t    0.01s
  extreme_points   util  18.11%  placed 100.00%  AE 0.33  SS 1.00  t    0.01s
  baf              util  18.11%  placed 100.00%  AE 0.17  SS 1.00  t    0.01s
  bssf             util  18.11%  placed 100.00%  AE 0.25  SS 1.00  t    0.01s
  blsf             util  18.11%  placed 100.00%  AE 0.17  SS 1.00  t    0.02s
  ga               util  18.11%  placed 100.00%  AE 0.33  SS 1.00  t    9.27s
  cpsat            util  18.11%  placed 100.00%  AE 0.42  SS 0.92  t    0.47s [OPTIMAL]
  pct              util  18.11%  placed 100.00%  AE 0.17  SS 1.00  t    0.20s
  morl_pct         util  18.11%  placed 100.00%  AE 0.33  SS 1.00  t    0.04s
-- seed 7 --
  bl               util  18.11%  placed 100.00%  AE 0.33  SS 1.00  t    0.01s
  extreme_points   ut

### 8a. Per-suite breakdown

For each suite, show the mean of each metric across seeds, sorted by utilisation. The winner column flags the best algorithm per metric.

In [4]:
import pandas as pd

df = pd.DataFrame(eval_rows)
print('Master CSV had', len(df), 'rows. Suites:', sorted(df['suite'].unique()))

for suite in SUITES_TO_RUN:
    sub = df[df.suite == suite]
    if sub.empty:
        continue
    agg = sub.groupby('algorithm').agg(
        util_mean=('util_pct', 'mean'),
        util_std=('util_pct', 'std'),
        placed_mean=('placed_pct', 'mean'),
        access_eff=('access_eff', 'mean'),
        stability=('stability_score', 'mean'),
        cog=('cog_long_abs', 'mean'),
        time_s=('elapsed_s', 'mean'),
    ).round(2).sort_values('util_mean', ascending=False)
    print(f'\n────── {suite} ──────')
    print(agg.to_string())
    winners = {
        'util':       agg['util_mean'].idxmax(),
        'placed':     agg['placed_mean'].idxmax(),
        'access':     agg['access_eff'].idxmax(),
        'stability':  agg['stability'].idxmax(),
        '|cog|':      agg['cog'].idxmin(),
        'fastest':    agg['time_s'].idxmin(),
    }
    print('  winners:', ' '.join(f'{k}={v}' for k, v in winners.items()))

Master CSV had 162 rows. Suites: ['20gp_tight', 'hazmat_heavy', 'large_mixed', 'medium_mixed', 'reefer', 'small_easy']

────── small_easy ──────
                util_mean  util_std  placed_mean  access_eff  stability   cog  time_s
algorithm                                                                            
baf                 18.11       0.0        100.0        0.17       1.00  0.03    0.01
bl                  18.11       0.0        100.0        0.33       1.00  0.20    0.01
blsf                18.11       0.0        100.0        0.17       1.00  0.06    0.01
bssf                18.11       0.0        100.0        0.25       1.00  0.00    0.01
cpsat               18.11       0.0        100.0        0.31       0.97  0.13    0.21
extreme_points      18.11       0.0        100.0        0.33       1.00  0.19    0.01
ga                  18.11       0.0        100.0        0.33       1.00  0.20    8.77
morl_pct            18.11       0.0        100.0        0.33       1.00  0.20    

### 8b. Overall ranking + head-to-head vs CP-SAT

Across all suites/seeds: mean metrics + how often each algorithm beats CP-SAT on utilisation.

In [5]:
# Overall mean across every (suite, seed)
overall = df.groupby('algorithm').agg(
    util_mean=('util_pct', 'mean'),
    util_std=('util_pct', 'std'),
    placed_mean=('placed_pct', 'mean'),
    access=('access_eff', 'mean'),
    stability=('stability_score', 'mean'),
    cog=('cog_long_abs', 'mean'),
    time_s=('elapsed_s', 'mean'),
).round(2).sort_values('util_mean', ascending=False)
print('=== OVERALL (all suites × seeds) ===')
print(overall.to_string())

# Head-to-head vs CP-SAT on utilisation, per (suite, seed)
print('\n=== HEAD-TO-HEAD vs CP-SAT (wins / ties / losses on util%) ===')
pivot = df.pivot_table(index=['suite', 'seed'], columns='algorithm', values='util_pct')
if 'cpsat' in pivot.columns:
    cps = pivot['cpsat']
    summary = {}
    for col in pivot.columns:
        if col == 'cpsat':
            continue
        diff = pivot[col] - cps
        summary[col] = dict(
            wins=int((diff > 0.5).sum()),
            ties=int(diff.abs().le(0.5).sum()),
            losses=int((diff < -0.5).sum()),
            mean_gap=round(diff.mean(), 2),
        )
    h2h = pd.DataFrame(summary).T[['wins', 'ties', 'losses', 'mean_gap']]
    print(h2h.to_string())
    print('\n(0.5pp tolerance for ties. mean_gap = algorithm_util − cpsat_util, '
          'so a negative value means CP-SAT typically wins by that much.)')
else:
    print('No CP-SAT rows in df — cannot compute head-to-head.')

=== OVERALL (all suites × seeds) ===
                util_mean  util_std  placed_mean  access  stability   cog  time_s
algorithm                                                                        
pct                 20.48     16.83        79.44    0.43       0.98  0.12    0.35
morl_pct            20.02     17.16        76.30    0.55       0.93  0.25    0.37
extreme_points      19.68     18.02        74.26    0.56       0.93  0.24    0.26
ga                  19.60     17.54        74.07    0.58       0.94  0.25  324.32
bl                  19.60     17.54        74.07    0.58       0.94  0.25    0.25
blsf                19.38     17.49        78.33    0.33       0.94  0.11    0.42
cpsat               19.20     17.37        78.80    0.49       0.91  0.22    5.67
bssf                18.93     17.52        77.96    0.53       0.90  0.23    0.23
baf                 18.02     15.05        76.76    0.41       0.91  0.15    0.27

=== HEAD-TO-HEAD vs CP-SAT (wins / ties / losses on util%) =

## 9. MORL-PCT preference sweep (optional)

If `MORL_CKPT` is set, sweep several preference vectors on the `medium_mixed` suite to inspect the Pareto trade-off the conditioned policy is producing. Each row is one preference; columns are the realised metrics.

In [6]:
# Preferences ordered: [utilisation, access, stability, cog, weight].
# Each row sums to ~1; the conditioned policy should bend metrics in the direction of the dial.
PREF_SWEEP = [
    ('util_max',      [0.80, 0.05, 0.05, 0.05, 0.05]),
    ('balanced',      [0.40, 0.20, 0.20, 0.10, 0.10]),
    ('access_lean',   [0.30, 0.50, 0.10, 0.05, 0.05]),
    ('stability_lean',[0.30, 0.10, 0.50, 0.05, 0.05]),
    ('cog_lean',      [0.30, 0.10, 0.10, 0.45, 0.05]),
]
SWEEP_SUITE = 'medium_mixed'
SWEEP_SEEDS = SEEDS[:2]  # 2 seeds keeps it quick

if not (MORL_CKPT and pathlib.Path(MORL_CKPT).exists()):
    print('Skipped: MORL_CKPT not set or file not found.')
else:
    sweep_rows = []
    for label, pref in PREF_SWEEP:
        for seed in SWEEP_SEEDS:
            cont, items = build_voyage(SWEEP_SUITE, seed)
            m = run_one('morl_pct', {'weights_path': MORL_CKPT, 'preference': pref}, cont, items)
            sweep_rows.append({'preference': label, 'seed': seed, **m})
            print(f'  {label:<16} seed={seed}  util={m["util_pct"]:.2f}  '
                  f'AE={m["access_eff"]:.2f}  SS={m["stability_score"]:.2f}  '
                  f'|cog|={m["cog_long_abs"]:.3f}  t={m["elapsed_s"]:.2f}s')

    sweep_df = pd.DataFrame(sweep_rows)
    print('\n=== MORL preference sweep — mean over seeds ===')
    sweep_agg = sweep_df.groupby('preference').agg(
        util=('util_pct', 'mean'),
        placed=('placed_pct', 'mean'),
        access=('access_eff', 'mean'),
        stability=('stability_score', 'mean'),
        cog=('cog_long_abs', 'mean'),
    ).round(3)
    # Re-order rows to match PREF_SWEEP definition order
    sweep_agg = sweep_agg.reindex([lbl for lbl, _ in PREF_SWEEP])
    print(sweep_agg.to_string())
    print('\nInterpretation: each row dials one objective. If MORL truly conditions on '
          'preference, util_max should top util, access_lean should top access, etc. '
          'If every row looks identical, the FiLM gating isn\'t reading the preference.')
    sweep_csv = out_dir / f"morl_pref_sweep_{time.strftime('%Y%m%d_%H%M%S')}.csv"
    sweep_df.to_csv(sweep_csv, index=False)
    print(f'→ wrote {sweep_csv}')

  util_max         seed=42  util=11.76  AE=0.42  SS=0.92  |cog|=0.335  t=0.64s
  util_max         seed=7  util=8.50  AE=0.78  SS=0.91  |cog|=0.332  t=0.43s
  balanced         seed=42  util=11.76  AE=0.42  SS=0.92  |cog|=0.335  t=0.58s
  balanced         seed=7  util=8.50  AE=0.78  SS=0.91  |cog|=0.332  t=0.25s
  access_lean      seed=42  util=11.76  AE=0.42  SS=0.92  |cog|=0.335  t=0.41s
  access_lean      seed=7  util=8.50  AE=0.78  SS=0.91  |cog|=0.332  t=0.24s
  stability_lean   seed=42  util=11.76  AE=0.42  SS=0.92  |cog|=0.335  t=0.26s
  stability_lean   seed=7  util=8.50  AE=0.78  SS=0.91  |cog|=0.332  t=0.23s
  cog_lean         seed=42  util=11.76  AE=0.42  SS=0.92  |cog|=0.335  t=0.27s
  cog_lean         seed=7  util=8.50  AE=0.78  SS=0.91  |cog|=0.332  t=0.23s

=== MORL preference sweep — mean over seeds ===
                  util  placed  access  stability    cog
preference                                              
util_max        10.131  81.667   0.603      0.918  0.333


---

## 10. Expanded benchmark v2 — challenging + realistic-dense suites

v3 splits the suites into two groups:

**Mixed challenging suites** (9) — Wadaboa+presets mixed pool, designed to discriminate algorithms. Util is intentionally low/medium because the Wadaboa real-item pool is dominated by small fillers.

| suite | items | container | content |
|---|---:|---|---|
| `medium_mixed` | 30 | 40HC | mixed |
| `large_mixed` | 60 | 40HC | mixed |
| `xl_mixed` | 100 | 40HC | mixed (selection pressure) |
| `xxl_mixed` | 150 | 40HC | mixed (extreme) |
| `20gp_tight` | 20 | 20GP | mixed |
| `40gp_medium` | 40 | 40GP | mixed |
| `45hc_xl` | 80 | 45HC | mixed |
| `carton_chaos` | 80 | 40HC | small cartons + crates + bags |
| `size_extremes` | 40 | 40HC | machinery + IBCs ⊕ cartons + bags |

**Realistic dense suites** (8) — every item is a real Alexandria-port cargo type at realistic counts. Cargo volume is 46–68 % of container volume, so a competent algorithm should hit **high util AND high placed %**.

| suite | items | container | content | est. cargo vol / capacity |
|---|---:|---|---|---|
| `dense_eur` | 40 (28 light + 12 heavy) | 40HC | EUR pallets | 46.1 / 76.4 m³ ≈ 60 % |
| `dense_us` | 28 | 40HC | US pallets | 41.6 / 76.4 ≈ 54 % |
| `dense_wooden` | 40 | 40HC | wooden crates | 48.0 / 76.4 ≈ 63 % |
| `dense_ibc` | 25 | 40HC | 1000 L IBCs | 34.8 / 76.4 ≈ 46 % |
| `dense_cartons_large` | 180 | 40HC | large cartons | 51.8 / 76.4 ≈ 68 % |
| `dense_reefer` | 18 | 40RF | reefer fruit pallets | 34.6 / 60.0 ≈ 58 % |
| `mixed_palletized` | 38 (12 EUR light + 8 heavy + 18 US) | 40HC | mixed pallets | 49.8 / 76.4 ≈ 65 % |
| `agri_export` | 90 (30 wooden + 60 grain bags) | 40HC | crates + bagged grain | 39.8 / 76.4 ≈ 52 % |

**Tip**: to run only the dense suites, set in the config cell below:
```python
V2_SUITES_TO_RUN = ['dense_eur', 'dense_us', 'dense_wooden', 'dense_ibc',
                    'dense_cartons_large', 'dense_reefer',
                    'mixed_palletized', 'agri_export']
```

`RUN_GA` defaults to `False` — GA is byte-identical to BL across every v1 voyage.

In [ ]:
# ========== v2 CONFIG ==========
# PCT_CKPT / MORL_CKPT / MORL_PREFERENCE were set in §8 config — re-set here if running v2 standalone.
try:
    PCT_CKPT, MORL_CKPT
except NameError:
    PCT_CKPT  = '/content/pct_latest.pt'
    MORL_CKPT = '/content/morl_pct_latest.pt'
    MORL_PREFERENCE = [0.7, 0.1, 0.1, 0.05, 0.05]

V2_SEEDS         = [42, 7, 123, 2024, 999]     # 5 seeds for tighter stats
V2_CPSAT_TIME    = 45.0                        # bumped from 30s — larger problems need it
V2_CPSAT_WORKERS = 4
V2_RUN_GA        = False                       # GA == BL exactly in v1 (18/18) — skip
V2_SUITES_TO_RUN = [
    # ----- challenging mixed-strategy suites (9) -----
    'medium_mixed', 'large_mixed', 'xl_mixed', 'xxl_mixed',
    '20gp_tight', '40gp_medium', '45hc_xl',
    'carton_chaos', 'size_extremes',
    # ----- realistic dense voyages (8) — high util+placed expected -----
    'dense_eur', 'dense_us', 'dense_wooden', 'dense_ibc',
    'dense_cartons_large', 'dense_reefer',
    'mixed_palletized', 'agri_export',
]
# ===============================

import os, sys, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand
        os.chdir(REPO_DIR)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        break

# Plotting deps
import subprocess
try:
    import matplotlib  # noqa: F401
    import seaborn  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib', 'seaborn'])

# Sanity check torch if checkpoints present
if (PCT_CKPT and pathlib.Path(PCT_CKPT).exists()) or (MORL_CKPT and pathlib.Path(MORL_CKPT).exists()):
    try:
        import torch  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])

print('REPO_DIR    =', REPO_DIR)
print('PCT_CKPT    =', PCT_CKPT, '(exists)' if PCT_CKPT and pathlib.Path(PCT_CKPT).exists() else '(SKIP)')
print('MORL_CKPT   =', MORL_CKPT, '(exists)' if MORL_CKPT and pathlib.Path(MORL_CKPT).exists() else '(SKIP)')
print('V2_SEEDS    =', V2_SEEDS)
print('V2_SUITES   =', V2_SUITES_TO_RUN)
print('CPSAT_TIME  =', V2_CPSAT_TIME, 's   workers =', V2_CPSAT_WORKERS)
print('RUN_GA      =', V2_RUN_GA)

In [ ]:
import random
from app.catalog.loader import get_container, get_cargo_preset
from app.data.alexandria_sampler import AlexandriaSampler, SamplerConfig

SUITE_SPECS_V2 = {
    # ---- challenging mixed-strategy suites ----
    'medium_mixed':  dict(items=30,  container='40HC', kind='sampler', strategy='mixed'),
    'large_mixed':   dict(items=60,  container='40HC', kind='sampler', strategy='mixed'),
    'xl_mixed':      dict(items=100, container='40HC', kind='sampler', strategy='mixed'),
    'xxl_mixed':     dict(items=150, container='40HC', kind='sampler', strategy='mixed'),
    '20gp_tight':    dict(items=20,  container='20GP', kind='sampler', strategy='mixed'),
    '40gp_medium':   dict(items=40,  container='40GP', kind='sampler', strategy='mixed'),
    '45hc_xl':       dict(items=80,  container='45HC', kind='sampler', strategy='mixed'),
    'carton_chaos':  dict(items=80,  container='40HC', kind='carton_chaos'),
    'size_extremes': dict(items=40,  container='40HC', kind='size_extremes'),
    # ---- realistic dense voyages (preset_mix) ----
    'dense_eur':           dict(container='40HC', kind='preset_mix',
                                mix={'eur_pallet_light': 28, 'eur_pallet_heavy': 12}),
    'dense_us':            dict(container='40HC', kind='preset_mix',
                                mix={'us_pallet': 28}),
    'dense_wooden':        dict(container='40HC', kind='preset_mix',
                                mix={'wooden_crate': 40}),
    'dense_ibc':           dict(container='40HC', kind='preset_mix',
                                mix={'ibc_1000l': 25}),
    'dense_cartons_large': dict(container='40HC', kind='preset_mix',
                                mix={'carton_large': 180}),
    'dense_reefer':        dict(container='40RF', kind='preset_mix',
                                mix={'reefer_fruit_pallet': 18}),
    'mixed_palletized':    dict(container='40HC', kind='preset_mix',
                                mix={'eur_pallet_light': 12, 'eur_pallet_heavy': 8, 'us_pallet': 18}),
    'agri_export':         dict(container='40HC', kind='preset_mix',
                                mix={'wooden_crate': 30, 'bagged_grain_50kg': 60}),
}

# Carton-only mix — items are roughly cardboard cartons & bags, no pallets
CARTON_MIX = ['carton_small', 'carton_medium', 'carton_large', 'wooden_crate', 'bagged_grain_50kg']

# Big items interleaved with tiny ones — high size-variance
SIZE_BIG_POOL   = ['machinery_small', 'ibc_1000l', 'eur_pallet_heavy', 'us_pallet']
SIZE_SMALL_POOL = ['carton_small', 'bagged_grain_50kg']


def build_voyage_v2(suite_name, seed):
    cfg = SUITE_SPECS_V2[suite_name]
    cont = get_container(cfg['container'])
    kind = cfg['kind']

    if kind == 'carton_chaos':
        rng = random.Random(seed)
        items = [
            get_cargo_preset(rng.choice(CARTON_MIX), item_id=f'cc-{seed}-{i:03d}')
            for i in range(cfg['items'])
        ]

    elif kind == 'size_extremes':
        rng = random.Random(seed)
        items = []
        for i in range(cfg['items']):
            pool = SIZE_BIG_POOL if (i % 3 == 0) else SIZE_SMALL_POOL
            items.append(get_cargo_preset(rng.choice(pool), item_id=f'sx-{seed}-{i:03d}'))

    elif kind == 'preset_mix':
        # Deterministic counts per preset, shuffled by seed so packing order isn't trivial.
        rng = random.Random(seed)
        items = []
        idx = 0
        for preset_code, count in cfg['mix'].items():
            for _ in range(count):
                items.append(get_cargo_preset(preset_code, item_id=f'{suite_name[:8]}-{seed}-{idx:03d}'))
                idx += 1
        rng.shuffle(items)

    else:  # sampler
        s = AlexandriaSampler(SamplerConfig(
            n_items=cfg['items'], strategy=cfg['strategy'], seed=seed,
        ))
        items = s.sample()
    return cont, items


# Sanity-check every suite builds cleanly on seed 42
for name in V2_SUITES_TO_RUN:
    cont, items = build_voyage_v2(name, V2_SEEDS[0])
    total_vol = sum(
        it.dimensions.length_mm * it.dimensions.width_mm * it.dimensions.height_mm
        for it in items
    ) / 1e9
    cap_vol = cont.internal.length_mm * cont.internal.width_mm * cont.internal.height_mm / 1e9
    print(f'{name:<22} container={cont.code.value:<5} items={len(items):>3}  '
          f'cargo_vol={total_vol:>6.2f} m³  cap={cap_vol:>5.2f} m³  vol_ratio={total_vol/cap_vol*100:>5.1f}%')

### 10b. Run the v2 benchmark

Expected wall-time on Colab CPU (RUN_GA off):
- Heuristics: <2 min total across all 45 voyages
- CP-SAT @ 45s budget × 45 voyages ≈ **~34 min** worst case (most return earlier)
- PCT + MORL inference: 2–4 min total
- **Total: ~40 min**

If you want to prototype, trim `V2_SEEDS = [42]` and `V2_SUITES_TO_RUN = ['medium_mixed', 'large_mixed', 'xl_mixed']` for a ~10-min sanity run first.

In [ ]:
import time, csv
from app.algorithms import get_algorithm
from app.algorithms.base import solve


def _ae_v2(placements, container):
    if not placements:
        return 0.0
    door = 0.20 * container.internal.length_mm
    return sum(
        1 for p in placements
        if (p.position.x_mm + p.rotated_dimensions.length_mm / 2) <= door
    ) / len(placements)


def run_one_v2(code, kwargs, cont, items):
    algo = get_algorithm(code, **kwargs)
    t0 = time.perf_counter()
    if hasattr(algo, 'prepare'):
        algo.prepare(cont, items)
    res, _ = solve(algorithm=algo, container=cont, items=items)
    elapsed = time.perf_counter() - t0
    k = res.kpis
    return {
        'util_pct': 100 * k.utilization,
        'placed_pct': 100 * len(res.placements) / max(len(items), 1),
        'access_eff': _ae_v2(res.placements, cont),
        'stability_score': max(0.0, (len(res.placements) - k.unstable_count) / max(len(res.placements), 1)),
        'cog_long_abs': abs(k.cog_long_dev),
        'weight_pct': 100 * k.weight_used,
        'elapsed_s': elapsed,
        'cpsat_status': str(algo.meta.get('cpsat_status', '')) if code == 'cpsat' else '',
    }


def _v2_algo_specs():
    specs = [
        ('bl', {}),
        ('extreme_points', {}),
        ('baf', {}),
        ('bssf', {}),
        ('blsf', {}),
        ('cpsat', {'time_limit_s': V2_CPSAT_TIME, 'num_search_workers': V2_CPSAT_WORKERS, 'enforce_imdg': True}),
    ]
    if V2_RUN_GA:
        specs.insert(5, ('ga', {}))
    if PCT_CKPT and pathlib.Path(PCT_CKPT).exists():
        specs.append(('pct', {'weights_path': PCT_CKPT}))
    if MORL_CKPT and pathlib.Path(MORL_CKPT).exists():
        specs.append(('morl_pct', {'weights_path': MORL_CKPT, 'preference': MORL_PREFERENCE}))
    return specs


v2_algo_specs = _v2_algo_specs()
print('Algorithms:', [c for c, _ in v2_algo_specs])
print('Total runs :', len(V2_SUITES_TO_RUN) * len(V2_SEEDS) * len(v2_algo_specs))

v2_rows = []
t_global = time.perf_counter()
for suite_name in V2_SUITES_TO_RUN:
    spec = SUITE_SPECS_V2[suite_name]
    print(f'\n========== {suite_name}  ({spec["container"]}, {spec["items"]} items) ==========')
    for seed in V2_SEEDS:
        cont, items = build_voyage_v2(suite_name, seed)
        print(f'-- seed {seed} --')
        for code, kwargs in v2_algo_specs:
            try:
                m = run_one_v2(code, kwargs, cont, items)
            except Exception as e:
                print(f'  {code:<16} FAILED: {type(e).__name__}: {e}')
                continue
            row = {'suite': suite_name, 'seed': seed, 'algorithm': code, **m}
            v2_rows.append(row)
            print(
                f'  {code:<16} util {m["util_pct"]:>6.2f}%  placed {m["placed_pct"]:>6.2f}%  '
                f'AE {m["access_eff"]:>4.2f}  SS {m["stability_score"]:>4.2f}  '
                f't {m["elapsed_s"]:>7.2f}s'
            )

print(f'\nTotal v2 eval wall time: {(time.perf_counter() - t_global)/60:.1f} min')

out_dir = pathlib.Path(REPO_DIR, 'benchmarks/out')
out_dir.mkdir(parents=True, exist_ok=True)
v2_csv = out_dir / f"full_eval_v2_{time.strftime('%Y%m%d_%H%M%S')}.csv"
with v2_csv.open('w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=list(v2_rows[0].keys()))
    w.writeheader()
    w.writerows(v2_rows)
print(f'→ wrote {v2_csv}')

## 11. Analysis — overall, per-suite, head-to-head, statistical significance

Tables suitable for the paper's *Results* section.

In [ ]:
import pandas as pd
import numpy as np

dfv2 = pd.DataFrame(v2_rows)
print(f'v2 master DataFrame: {len(dfv2)} rows, '
      f'{dfv2.suite.nunique()} suites × {dfv2.seed.nunique()} seeds × {dfv2.algorithm.nunique()} algos')

# ---- Overall ranking ----
overall = dfv2.groupby('algorithm').agg(
    util_mean=('util_pct', 'mean'),
    util_std=('util_pct', 'std'),
    placed_mean=('placed_pct', 'mean'),
    access=('access_eff', 'mean'),
    stability=('stability_score', 'mean'),
    cog=('cog_long_abs', 'mean'),
    time_s=('elapsed_s', 'mean'),
).round(2).sort_values('util_mean', ascending=False)
print('\n=== OVERALL RANKING (all suites × seeds) ===')
print(overall.to_string())

# ---- Per-suite breakdown ----
print('\n\n=== PER-SUITE WINNERS ===')
winner_rows = []
for suite in V2_SUITES_TO_RUN:
    sub = dfv2[dfv2.suite == suite]
    if sub.empty:
        continue
    agg = sub.groupby('algorithm').agg(util=('util_pct','mean')).round(2)
    winner = agg['util'].idxmax()
    winner_rows.append({
        'suite': suite,
        'winner': winner,
        'winner_util': agg.loc[winner, 'util'],
        'runner_up': agg['util'].nlargest(2).index[1],
        'runner_up_util': agg['util'].nlargest(2).iloc[1],
        'gap': round(agg['util'].nlargest(2).iloc[0] - agg['util'].nlargest(2).iloc[1], 2),
    })
print(pd.DataFrame(winner_rows).to_string(index=False))

# ---- Head-to-head pivot ----
pivot = dfv2.pivot_table(index=['suite', 'seed'], columns='algorithm', values='util_pct')
print('\n=== HEAD-TO-HEAD on util_pct (rows = suite/seed, cols = algorithm) ===')
print(pivot.round(2).to_string())

In [ ]:
# ---- Pairwise win-count matrix + paired Wilcoxon p-values ----
from itertools import combinations

algos = sorted(dfv2['algorithm'].unique())
win_matrix = pd.DataFrame(0, index=algos, columns=algos, dtype=int)
for (s, sd), grp in dfv2.groupby(['suite', 'seed']):
    g = grp.set_index('algorithm')['util_pct']
    for a in algos:
        for b in algos:
            if a == b or a not in g.index or b not in g.index:
                continue
            if g[a] > g[b] + 0.5:    # 0.5 pp tolerance for ties
                win_matrix.loc[a, b] += 1

print('=== WIN-COUNT MATRIX (row beats column on util_pct, 0.5pp tolerance) ===')
print(win_matrix.to_string())

# Paired Wilcoxon vs the overall winner
try:
    from scipy.stats import wilcoxon
    leader = overall.index[0]
    print(f'\n=== PAIRED WILCOXON vs LEADER ({leader}) on util_pct ===')
    print(f'{"opponent":<16} {"median_diff":>11} {"p_value":>9} {"n_pairs":>7}')
    for opp in algos:
        if opp == leader:
            continue
        sub = dfv2[dfv2['algorithm'].isin([leader, opp])]
        wide = sub.pivot_table(index=['suite', 'seed'], columns='algorithm', values='util_pct').dropna()
        if len(wide) < 5:
            continue
        diffs = wide[leader] - wide[opp]
        try:
            stat, p = wilcoxon(diffs, alternative='greater')  # leader > opp
            print(f'{opp:<16} {diffs.median():>11.2f} {p:>9.4f} {len(diffs):>7}')
        except ValueError:
            print(f'{opp:<16} {diffs.median():>11.2f} {"(zero diff)":>9} {len(diffs):>7}')
except ImportError:
    print('scipy not installed — skipping Wilcoxon')
    !pip install -q scipy

---

## 12. Figures for the paper

All figures saved as **vector PDF** (paper-ready) and **PNG** (inline preview) into `benchmarks/figures/`. Each cell is self-contained and re-runnable.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import numpy as np
import pandas as pd
import pathlib

# Paper-ready style — large fonts, vector output, no gridlines on bars
mpl.rcParams.update({
    'figure.figsize': (8, 5),
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'pdf.fonttype': 42,  # editable text in vector PDFs
    'ps.fonttype': 42,
})
sns.set_palette('deep')

FIG_DIR = pathlib.Path(REPO_DIR, 'benchmarks/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Algorithm display names + colour map (consistent across all figures)
ALGO_LABEL = {
    'bl': 'BL', 'extreme_points': 'EP', 'baf': 'BAF', 'bssf': 'BSSF', 'blsf': 'BLSF',
    'ga': 'GA', 'cpsat': 'CP-SAT', 'pct': 'PCT', 'morl_pct': 'MORL-PCT',
}
PALETTE = dict(zip(
    ['bl', 'extreme_points', 'baf', 'bssf', 'blsf', 'ga', 'cpsat', 'pct', 'morl_pct'],
    sns.color_palette('deep', 9),
))


def savefig(name):
    """Save current figure as PDF + PNG into FIG_DIR."""
    plt.tight_layout()
    for ext in ('pdf', 'png'):
        plt.savefig(FIG_DIR / f'{name}.{ext}')
    print(f'  saved {name}.{{pdf,png}}')


print('FIG_DIR =', FIG_DIR)

In [ ]:
# ---------- Figure 1: overall ranking on util_pct with error bars ----------
order = overall.index.tolist()
means = overall['util_mean'].values
stds  = overall['util_std'].values

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = [PALETTE[a] for a in order]
ax.bar(range(len(order)), means, yerr=stds, capsize=4, color=colors, edgecolor='black', linewidth=0.6)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([ALGO_LABEL.get(a, a) for a in order], rotation=20)
ax.set_ylabel('Mean utilisation %')
ax.set_title('Overall utilisation by algorithm — mean ± std across 9 suites × 5 seeds')
ax.grid(axis='y', linestyle=':', alpha=0.5)
ax.set_axisbelow(True)
# Annotate bars with values
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.4, f'{m:.1f}', ha='center', fontsize=10)
savefig('fig01_overall_util_bar')
plt.show()

In [ ]:
# ---------- Figure 2: heatmap suite × algorithm (mean util_pct) ----------
heat = dfv2.pivot_table(index='suite', columns='algorithm', values='util_pct', aggfunc='mean')
# Keep suite order from V2_SUITES_TO_RUN, algorithm order from overall ranking
heat = heat.reindex(index=V2_SUITES_TO_RUN, columns=order)

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlGnBu',
            cbar_kws={'label': 'util %'}, linewidths=0.4, ax=ax)
# Highlight per-row winners with a red border
for i, suite in enumerate(heat.index):
    j = int(np.argmax(heat.loc[suite].values))
    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='crimson', linewidth=2.2))
ax.set_title('Mean utilisation % per (suite × algorithm) — red box = per-suite winner')
ax.set_ylabel('Suite')
ax.set_xlabel('Algorithm')
ax.set_xticklabels([ALGO_LABEL.get(a, a) for a in heat.columns], rotation=20)
savefig('fig02_suite_algo_heatmap')
plt.show()

In [ ]:
# ---------- Figure 3: box plot util distribution per algorithm ----------
fig, ax = plt.subplots(figsize=(9, 4.5))
plot_df = dfv2.copy()
plot_df['algo_label'] = plot_df['algorithm'].map(ALGO_LABEL)
sns.boxplot(
    data=plot_df, x='algorithm', y='util_pct', order=order,
    palette=[PALETTE[a] for a in order], width=0.6, fliersize=3, linewidth=1.1, ax=ax,
)
sns.stripplot(
    data=plot_df, x='algorithm', y='util_pct', order=order,
    color='black', size=2.2, alpha=0.45, jitter=0.18, ax=ax,
)
ax.set_xticklabels([ALGO_LABEL.get(a, a) for a in order], rotation=20)
ax.set_xlabel('')
ax.set_ylabel('Utilisation % per voyage')
ax.set_title('Distribution of utilisation across 45 voyages — wider boxes ⇒ less consistent algorithm')
ax.grid(axis='y', linestyle=':', alpha=0.5)
ax.set_axisbelow(True)
savefig('fig03_util_boxplot')
plt.show()

In [ ]:
# ---------- Figure 4: Pareto scatter — util vs access_eff (per algorithm) ----------
mean_per_algo = dfv2.groupby('algorithm').agg(
    util=('util_pct', 'mean'),
    util_std=('util_pct', 'std'),
    access=('access_eff', 'mean'),
    access_std=('access_eff', 'std'),
).reindex(order)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
for a in order:
    ax.errorbar(
        mean_per_algo.loc[a, 'access'], mean_per_algo.loc[a, 'util'],
        xerr=mean_per_algo.loc[a, 'access_std'], yerr=mean_per_algo.loc[a, 'util_std'],
        fmt='o', color=PALETTE[a], ecolor=PALETTE[a], alpha=0.85,
        markersize=10, markeredgecolor='black', markeredgewidth=0.6,
        capsize=3, label=ALGO_LABEL.get(a, a),
    )
    ax.annotate(ALGO_LABEL.get(a, a),
                xy=(mean_per_algo.loc[a, 'access'], mean_per_algo.loc[a, 'util']),
                xytext=(6, 4), textcoords='offset points', fontsize=10)
ax.set_xlabel('Mean access efficiency (door-zone share)')
ax.set_ylabel('Mean utilisation %')
ax.set_title('Pareto view: utilisation vs access efficiency (up-right is better)')
ax.grid(linestyle=':', alpha=0.5)
savefig('fig04_pareto_util_vs_access')
plt.show()

In [ ]:
# ---------- Figure 5: pairwise win-count heatmap ----------
wm = win_matrix.reindex(index=order, columns=order)
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(wm, annot=True, fmt='d', cmap='RdYlGn',
            cbar_kws={'label': '# voyages row beats column on util'},
            linewidths=0.4, ax=ax, vmin=0, vmax=wm.values.max())
ax.set_title('Pairwise win-count (rows beat columns on util_pct, 0.5pp tolerance)')
ax.set_xticklabels([ALGO_LABEL.get(a, a) for a in order], rotation=30, ha='right')
ax.set_yticklabels([ALGO_LABEL.get(a, a) for a in order], rotation=0)
ax.set_xlabel('Opponent (column)')
ax.set_ylabel('Algorithm (row)')
savefig('fig05_win_heatmap')
plt.show()

In [ ]:
# ---------- Figure 6: scaling — util vs item count ----------
# Restrict to the 40HC scaling-only suites where item count varies on a fixed container
SCALING_SUITES = ['medium_mixed', 'large_mixed', 'xl_mixed', 'xxl_mixed']
sc = dfv2[dfv2['suite'].isin(SCALING_SUITES)].copy()
sc['n_items'] = sc['suite'].map({k: SUITE_SPECS_V2[k]['items'] for k in SCALING_SUITES})
scale_agg = sc.groupby(['algorithm', 'n_items'])['util_pct'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
for a in order:
    sub = scale_agg[scale_agg['algorithm'] == a].sort_values('n_items')
    if sub.empty:
        continue
    ax.errorbar(sub['n_items'], sub['mean'], yerr=sub['std'],
                fmt='-o', color=PALETTE[a], markersize=7,
                markeredgecolor='black', markeredgewidth=0.5,
                capsize=3, label=ALGO_LABEL.get(a, a), linewidth=1.6)
ax.set_xlabel('Items per voyage (40HC)')
ax.set_ylabel('Mean utilisation %')
ax.set_title('Scaling: utilisation as item count grows (40HC, mixed strategy)')
ax.legend(ncol=3, loc='upper left', fontsize=10, framealpha=0.92)
ax.grid(linestyle=':', alpha=0.5)
savefig('fig06_util_scaling')
plt.show()

In [ ]:
# ---------- Figure 7: speed vs quality (time × util scatter) ----------
fig, ax = plt.subplots(figsize=(8, 5.5))
algo_time = dfv2.groupby('algorithm')['elapsed_s'].mean().reindex(order)
algo_util = overall['util_mean']
algo_util_std = overall['util_std']

for a in order:
    ax.errorbar(
        algo_time[a], algo_util[a], yerr=algo_util_std[a],
        fmt='o', color=PALETTE[a], markersize=12, markeredgecolor='black',
        markeredgewidth=0.6, capsize=3, label=ALGO_LABEL.get(a, a),
    )
    ax.annotate(ALGO_LABEL.get(a, a),
                xy=(algo_time[a], algo_util[a]),
                xytext=(7, 5), textcoords='offset points', fontsize=10)
ax.set_xscale('log')
ax.set_xlabel('Mean wall-clock per voyage (s, log scale)')
ax.set_ylabel('Mean utilisation %')
ax.set_title('Quality vs speed — top-left wins (high util, low time)')
ax.grid(linestyle=':', alpha=0.5, which='both')
savefig('fig07_speed_vs_quality')
plt.show()

In [ ]:
# ---------- Figure 8: multi-metric radar (mean util / placed / access / stability / 1−|cog|) ----------
metrics = ['util_pct', 'placed_pct', 'access_eff', 'stability_score', 'cog_long_abs']
labels  = ['Util %', 'Placed %', 'Access', 'Stability', '1 − |CoG|']

# Normalise each metric to [0, 1] across algorithms (so radar shows relative strength)
norm = dfv2.groupby('algorithm')[metrics].mean()
norm['cog_long_abs'] = 1.0 - norm['cog_long_abs']  # higher = better
norm = norm.reindex(order)
mn, mx = norm.min(), norm.max()
norm_scaled = (norm - mn) / (mx - mn + 1e-9)

angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

# Only show top-5 algorithms to keep the radar readable
top5 = overall.head(5).index.tolist()
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})
for a in top5:
    vals = norm_scaled.loc[a].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, '-o', color=PALETTE[a], linewidth=1.7, label=ALGO_LABEL.get(a, a))
    ax.fill(angles, vals, color=PALETTE[a], alpha=0.10)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_title('Multi-metric profile (top-5 algorithms, normalised across the board)', pad=24)
ax.legend(loc='upper right', bbox_to_anchor=(1.30, 1.05), fontsize=10)
savefig('fig08_multimetric_radar')
plt.show()

---

## 13. CP-SAT time-budget sweep

Does more time help CP-SAT? Run the same `large_mixed` voyage at five different time limits and plot the convergence curve. Useful to defend the choice of `time_limit_s=45` (or to show CP-SAT plateaus regardless).

In [ ]:
# CP-SAT time-budget sweep on large_mixed × all V2 seeds
BUDGETS = [5, 15, 30, 60, 120]    # seconds
SWEEP_SUITE = 'large_mixed'

budget_rows = []
for seed in V2_SEEDS:
    cont, items = build_voyage_v2(SWEEP_SUITE, seed)
    for budget in BUDGETS:
        m = run_one_v2('cpsat',
                       {'time_limit_s': float(budget), 'num_search_workers': V2_CPSAT_WORKERS, 'enforce_imdg': True},
                       cont, items)
        budget_rows.append({'seed': seed, 'budget_s': budget, **m})
        print(f'  seed={seed}  budget={budget:>3}s   util={m["util_pct"]:>6.2f}%   '
              f'wall={m["elapsed_s"]:>6.2f}s')

bdf = pd.DataFrame(budget_rows)
budget_csv = out_dir / f"cpsat_budget_sweep_{time.strftime('%Y%m%d_%H%M%S')}.csv"
bdf.to_csv(budget_csv, index=False)
print(f'\n→ wrote {budget_csv}')

# Plot
budget_agg = bdf.groupby('budget_s')['util_pct'].agg(['mean', 'std']).reset_index()
# Reference: PCT mean util on large_mixed for visual comparison
pct_ref = dfv2[(dfv2['suite']==SWEEP_SUITE) & (dfv2['algorithm']=='pct')]['util_pct'].mean()

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(budget_agg['budget_s'], budget_agg['mean'], yerr=budget_agg['std'],
            fmt='-o', color=PALETTE['cpsat'], markersize=9,
            markeredgecolor='black', markeredgewidth=0.5, capsize=4, linewidth=1.8,
            label='CP-SAT')
if not np.isnan(pct_ref):
    ax.axhline(pct_ref, ls='--', color=PALETTE['pct'], linewidth=1.5, label=f'PCT reference ({pct_ref:.2f}%)')
ax.set_xscale('log')
ax.set_xlabel('CP-SAT time budget (s, log scale)')
ax.set_ylabel('Mean utilisation % on large_mixed')
ax.set_title('Does CP-SAT improve with more time? (5 seeds averaged)')
ax.grid(linestyle=':', alpha=0.5, which='both')
ax.legend()
savefig('fig09_cpsat_budget_sweep')
plt.show()

---

## 14. Export everything as a paper-ready bundle

Packages every CSV in `benchmarks/out/` and every figure in `benchmarks/figures/` into a single zip you can download from Colab in one go.

In [ ]:
import shutil, zipfile, time

stamp = time.strftime('%Y%m%d_%H%M%S')
bundle = pathlib.Path(REPO_DIR, f'benchmarks/paper_bundle_{stamp}.zip')

with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zf:
    for d in ('benchmarks/out', 'benchmarks/figures'):
        for fp in pathlib.Path(REPO_DIR, d).glob('*'):
            zf.write(fp, arcname=str(fp.relative_to(pathlib.Path(REPO_DIR, 'benchmarks'))))

size_mb = bundle.stat().st_size / 1e6
print(f'→ wrote {bundle}   ({size_mb:.2f} MB)')
print('In Colab: right-click the file in the Files panel → Download.')